In [426]:

import numpy as np
import pandas as pd
import polars as pl
import scipy as sp
import sklearn
import matplotlib.pyplot as plt
import itertools

import os


np.random.seed(0)

In [427]:
ROOT = './data/'

# Dataset

## train.csv

Данный файл содержит информацию о том выйграла ли команда radiant в матче с идентификатором mid.

Каждая строчка файла содержит следующие колонки:

* mid — идентификатор матча
* radiant_won — факт того, что в игре победила команда radiant

In [428]:
train_table = pd.read_csv(ROOT + 'train.csv', index_col='mid')
train_table.head()

,radiant_won
mid,
0,1
1,0
2,1
4,1
5,1


In [429]:
len(train_table)

24974

In [430]:
train_table['radiant_won'].value_counts()

radiant_won
1    12974
0    12000
Name: count, dtype: int64

## test.csv
В данном файле содержатся идентификаторы игр для которых нужно сделать предсказание о победе команды radiant.

Каждая строчка файла содержит следующие колонки:

* mid — идентификатор матча

In [431]:
hidden_list = pd.read_csv(ROOT + 'test.csv').to_numpy().flatten()
hidden_list

array([    3,     7,     9, ..., 49943, 49944, 49947], dtype=int64)

In [432]:
len(hidden_list)

24974

**teams**

In [433]:
first_team_cols = [f'player_{i}' for i in range(0, 5)]
second_team_cols = [f'player_{i}' for i in range(5, 10)]
player_cols = first_team_cols + second_team_cols

In [434]:
def add_col_prefix(df: pd.DataFrame, col_prefix: str = "") -> pd.DataFrame:
    df.rename(columns={col_name : f'{col_prefix}_{col_name}' for col_name in df.columns}, inplace=True)
    return df

def create_player_time_stamp_features(df: pd.DataFrame, time_stemps: np.ndarray, player_cols: list[str], col_prefix: str = "",) -> pd.DataFrame:
    new_cols = {}
    for player, time in itertools.product(player_cols, time_stemps):
        mask = df['times'] == time
        new_cols[f"{col_prefix}_{player}_{time}"] = df[mask][player].to_numpy()
    return pd.DataFrame(new_cols, index = df.index.unique())

## gold.csv

Данный файл содержит снимки данных игры, содержащих количество золота для каждого игрока.

Каждая строчка файла содержит следующие колонки:

* mid — идентификатор матча
* times — время в секундах когда был сделан снимок
* player_0, player_1, player_2, player_3, player_4 — количество золота для игроков команды radiant
* player_5, player_6, player_7, player_8, player_9 — количество золота для игроков команды dire

In [435]:
gold_table = pd.read_csv(ROOT + 'gold.csv', index_col='mid')
gold_table.head()

,times,player_0,player_1,player_2,player_3,player_4,player_5,player_6,player_7,player_8,player_9
mid,,,,,,,,,,,
0,60,750,350,389,437,428,398,344,654,287,1056
0,120,957,1071,633,655,1080,669,1147,1164,438,1360
0,180,1161,1527,782,1103,1346,1058,1479,1574,587,2072
0,240,1571,2033,932,1515,2058,1760,1767,2387,737,2283
0,300,1721,2313,1082,1790,2699,2087,1986,2898,887,3302


In [436]:
mid = gold_table.index.unique().to_numpy()

In [437]:
time_stemps = np.unique(gold_table['times'])
print(*time_stemps)

60 120 180 240 300 360 420 480 540 600


In [438]:
gold_table = create_player_time_stamp_features(
    df=gold_table, 
    time_stemps=time_stemps, 
    player_cols=player_cols,
    col_prefix="gold"
)

In [439]:
gold_table.head()

,gold_player_0_60,gold_player_0_120,gold_player_0_180,gold_player_0_240,gold_player_0_300,gold_player_0_360,gold_player_0_420,gold_player_0_480,gold_player_0_540,gold_player_0_600,...,gold_player_9_60,gold_player_9_120,gold_player_9_180,gold_player_9_240,gold_player_9_300,gold_player_9_360,gold_player_9_420,gold_player_9_480,gold_player_9_540,gold_player_9_600
mid,,,,,,,,,,,,,,,,,,,,,
0,750,957,1161,1571,1721,1871,2022,2850,3303,3454,...,1056,1360,2072,2283,3302,4071,4686,5207,5609,6384
1,285,435,585,736,1334,1667,1818,2016,2328,2477,...,513,851,1239,1840,2052,2321,3214,3603,4062,4623
2,288,756,1224,1617,1920,2328,2611,2879,3069,3604,...,438,646,796,946,1168,1660,1810,1959,2218,2491
3,288,438,1230,1381,1916,2436,2585,2735,2886,3457,...,288,438,795,946,1340,1591,1740,1890,2097,2247
4,348,572,745,1170,1590,1787,2070,2520,2948,3675,...,288,437,643,855,1065,1499,1649,1800,2070,2220


## lh.csv

Данный файл содержит снимки данных, содержащих количество убитых нейтральных монстров (крипов) для каждого игрока.

Каждая строчка файла содержит следующие колонки:

* mid — идентификатор матча
* times — время в секундах когда был сделан снимок
* player_0, player_1, player_2, player_3, player_4 — количество убитых монстров для игроков команды radiant
* player_5, player_6, player_7, player_8, player_9 — количество убитых монстров для игроков команды dire

In [440]:
lh_table = pd.read_csv(ROOT + 'lh.csv', index_col='mid')
lh_table.head()

,times,player_0,player_1,player_2,player_3,player_4,player_5,player_6,player_7,player_8,player_9
mid,,,,,,,,,,,
0,60,1,2,1,1,2,3,2,7,1,2
0,120,1,5,1,2,6,5,6,14,1,6
0,180,2,10,1,7,8,9,9,18,1,9
0,240,2,13,1,13,12,9,12,29,1,10
0,300,2,15,1,17,19,13,13,36,1,19


In [441]:
lh_table = create_player_time_stamp_features(
    df=lh_table,
    time_stemps=time_stemps,
    player_cols=player_cols,
    col_prefix='lh'
)
lh_table.head()

,lh_player_0_60,lh_player_0_120,lh_player_0_180,lh_player_0_240,lh_player_0_300,lh_player_0_360,lh_player_0_420,lh_player_0_480,lh_player_0_540,lh_player_0_600,...,lh_player_9_60,lh_player_9_120,lh_player_9_180,lh_player_9_240,lh_player_9_300,lh_player_9_360,lh_player_9_420,lh_player_9_480,lh_player_9_540,lh_player_9_600
mid,,,,,,,,,,,,,,,,,,,,,
0,1,1,2,2,2,2,3,4,4,4,...,2,6,9,10,19,31,31,36,38,46
1,1,1,1,1,1,4,4,4,5,5,...,5,8,13,17,18,20,29,34,39,39
2,1,6,12,16,17,21,24,26,27,34,...,1,2,2,2,4,5,5,5,8,12
3,1,1,1,1,1,1,1,1,1,1,...,1,1,2,2,7,8,8,8,8,8
4,2,4,5,5,9,10,15,20,27,30,...,1,1,2,3,4,4,4,4,6,6


## xp.csv

Данный файл содержит снимки данных, содержащих количество заработанного опыта для каждого игрока.

Каждая строчка файла содержит следующие колонки:

* mid — идентификатор матча
* times — время в секундах когда был сделан снимок
* player_0, player_1, player_2, player_3, player_4 — количество заработанного опыта для игроков команды radiant
* player_5, player_6, player_7, player_8, player_9 — количество заработанного опыта для игроков команды dire

In [442]:
xp_table = pd.read_csv(ROOT + 'xp.csv', index_col='mid')
xp_table.head()

,times,player_0,player_1,player_2,player_3,player_4,player_5,player_6,player_7,player_8,player_9
mid,,,,,,,,,,,
0,60,79,214,147,222,147,94,78,396,94,147
0,120,321,719,423,777,421,490,607,895,241,365
0,180,356,1333,424,1300,638,922,937,1259,242,590
0,240,544,1752,441,1782,1348,1460,1163,2037,242,658
0,300,724,2002,565,2087,1807,2102,1498,2389,276,1020


In [443]:
xp_table = create_player_time_stamp_features(
    df=xp_table,
    time_stemps=time_stemps,
    player_cols=player_cols,
    col_prefix='xp'
)
xp_table.head()

,xp_player_0_60,xp_player_0_120,xp_player_0_180,xp_player_0_240,xp_player_0_300,xp_player_0_360,xp_player_0_420,xp_player_0_480,xp_player_0_540,xp_player_0_600,...,xp_player_9_60,xp_player_9_120,xp_player_9_180,xp_player_9_240,xp_player_9_300,xp_player_9_360,xp_player_9_420,xp_player_9_480,xp_player_9_540,xp_player_9_600
mid,,,,,,,,,,,,,,,,,,,,,
0,79,321,356,544,724,758,865,1259,1542,1696,...,147,365,590,658,1020,1382,1471,1699,1876,2347
1,45,326,517,677,1012,1092,1091,1194,1470,1542,...,305,736,1178,1564,1883,2109,2723,3001,3318,3662
2,124,475,848,1145,1519,1859,2017,2325,2478,2851,...,223,528,707,910,1137,1543,1691,1921,2181,2426
3,78,191,385,497,597,819,820,869,891,1145,...,100,289,485,541,841,1073,1255,1255,1452,1530
4,107,243,289,352,717,853,1114,1448,1741,2197,...,11,223,455,646,961,1280,1314,1611,1859,1924


## items.csv

Разреженная матрица содержащая информацию о предметах, которые успели купить игроки за первые 10 минут игры.

Каждая строчка файла содержит следующие колонки:

* mid — идентификатор матча
* player — идентификатор игрока, с 0 по 4 игроки команды radiant, с 5 по 9 игроки команды dire
* item_0, … , item_120 — количество купленных предметов

In [444]:
items_table = pd.read_csv(ROOT + 'items.csv', index_col='mid')
items_table.head()

,player,item_0,item_1,item_2,item_3,item_4,item_5,item_6,item_7,item_8,...,item_111,item_112,item_113,item_114,item_115,item_116,item_117,item_118,item_119,item_120
mid,,,,,,,,,,,,,,,,,,,,,
0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,3,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN


In [445]:
items_table_per_team = pd.DataFrame(items_table.drop('player', axis=1)).fillna(0)
add_col_prefix(items_table, col_prefix='items');

In [446]:
items_table_per_team = items_table_per_team.groupby('mid').apply(lambda x: np.sum(x, axis=0))

## heroes.csv

Данный файл содержит информацию о типах героев, которые были выбраны игроки в каждом матче.

Каждая строчка файла содержит следующие колонки:

* mid — идентификатор матча
* player_0, player_1, player_2, player_3, player_4 — типы героев, выбранных игроками команды radiant
* player_5, player_6, player_7, player_8, player_9 — типы героев, выбранных игроками команды dire

In [447]:
heroes_table = pd.read_csv(ROOT + 'heroes.csv', index_col='mid')
heroes_table.head()

,player_0,player_1,player_2,player_3,player_4,player_5,player_6,player_7,player_8,player_9
mid,,,,,,,,,,
0,91,42,87,15,65,11,6,34,69,74
1,69,85,71,24,64,74,68,39,65,11
2,17,40,31,67,99,32,7,72,48,104
3,80,43,101,71,94,69,70,98,24,39
4,25,15,75,29,95,3,32,55,64,86


In [448]:
heroes_table = add_col_prefix(heroes_table, col_prefix='heroes')
heroes_table.head()

,heroes_player_0,heroes_player_1,heroes_player_2,heroes_player_3,heroes_player_4,heroes_player_5,heroes_player_6,heroes_player_7,heroes_player_8,heroes_player_9
mid,,,,,,,,,,
0,91,42,87,15,65,11,6,34,69,74
1,69,85,71,24,64,74,68,39,65,11
2,17,40,31,67,99,32,7,72,48,104
3,80,43,101,71,94,69,70,98,24,39
4,25,15,75,29,95,3,32,55,64,86


In [449]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False)
heroes_table = ohe.fit_transform(heroes_table)

## events.csv

Данный файл содержит информацию о событиях, которые произошли в каждом матче.

Каждая строчка файла содержит следующие колонки:

* mid — идентификатор матча
* event_type — тип события
  * 0 — Командой был забран Aegis
  * 1 — Командой был украден Aegis
  * 2 — Командой были разрушены бараки соперника
  * 3 — Командой был сделано первое убийство героя соперника
  * 4 — Командой был убит Roshan.
  * 5 — Командой была разрушена своя башня
  * 6 — Командой была разрушена башня соперника
* from_team — название команды, которое инициировало событие
* time — время в секундах от старта игры когда событие произошло

In [450]:
events_table = pd.read_csv(ROOT + 'events.csv', index_col='mid').sort_index()
events_table.head(10)

,event_type,from_team,time
mid,,,
0,3,radiant,1
1,3,radiant,222
2,3,dire,143
3,3,radiant,143
4,3,dire,53
6,3,radiant,82
6,6,radiant,523
7,3,dire,77
7,6,radiant,283


In [451]:
events_table.drop('time', axis=1, inplace=True)

In [452]:
events = sorted(events_table['event_type'].unique())
teams = sorted(events_table['from_team'].unique())

In [455]:
events_flags_table = pd.DataFrame({
    'mid' : mid,
    **{f'{t}_event_{e}' : np.full(mid.shape, fill_value=0) for t in teams for e in events}
}).set_index('mid')

In [454]:
events_table = events_table.groupby(['mid', 'from_team']).agg(tuple)

In [456]:
events_table.head(10)

event_type
mid from_team           
0   radiant         (3,)
1   radiant         (3,)
2   dire            (3,)
3   radiant         (3,)
4   dire            (3,)
6   radiant       (3, 6)
7   dire            (3,)
    radiant       (6, 6)
8   dire          (3, 6)
9   dire            (3,)

In [459]:
for m, t in events_table.index:
    #print(events_table.loc[m, t])
    for e in events_table.loc[m, t].event_type:
        events_flags_table.loc[m, f'{t}_event_{e}'] += 1

In [461]:
events_flags_table.head(10)

,dire_event_0,dire_event_1,dire_event_2,dire_event_3,dire_event_4,dire_event_5,dire_event_6,radiant_event_0,radiant_event_1,radiant_event_2,radiant_event_3,radiant_event_4,radiant_event_5,radiant_event_6
mid,,,,,,,,,,,,,,
0,0,0,0,0,0,0,0,0,0,0,2,0,0,0
1,0,0,0,0,0,0,0,0,0,0,2,0,0,0
2,0,0,0,2,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,2,0,0,0
4,0,0,0,2,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,2,0,0,2
7,0,0,0,2,0,0,0,0,0,0,0,0,0,4
8,0,0,0,2,0,0,2,0,0,0,0,0,0,0


# Algorithms

In [462]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from sklearn.linear_model import Ridge, ElasticNet, LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report, make_scorer

In [463]:
dataset = np.concatenate([gold_table, lh_table, xp_table, heroes_table, items_table_per_team, events_flags_table], axis=1)

X_data = dataset[train_table.index]
y_data = train_table['radiant_won']

X_hidden = dataset[hidden_list]

In [69]:
#dataset = pd.concat([gold_table, lh_table, xp_table, heroes_table, ], axis=1)
#cat_cols = np.fromiter(map(lambda x: x.startswith('heroes'), dataset.columns.to_list()), dtype=bool)

# X_data = dataset.loc[train_table.index].to_numpy()
# y_data = train_table['radiant_won'].to_numpy()

# X_hidden = dataset.loc[hidden_list].to_numpy()

In [464]:
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, shuffle=True, stratify=y_data, random_state=0)

# Mixed Naive Bayes

In [71]:
# gaussian_nb = Pipeline([
#     ('std_scaler', StandardScaler()),  # only for numerical purposes
#     ('gnb', GaussianNB())
# ])

# gaussian_nb.fit(X_train[:, ~cat_cols], y_train)

# y_pred_proba_gaussian_nb = gaussian_nb.predict_proba(X_test[:, ~cat_cols])

# cat_nb = CategoricalNB()
# cat_nb.fit(X_train[:, cat_cols], y_train)

# y_pred_proba_cat_nb = cat_nb.predict_proba(X_test[:, cat_cols])

# y_pred_mixed_nb = y_pred_proba_gaussian_nb * y_pred_proba_cat_nb / gaussian_nb['gnb'].class_prior_

In [72]:
# roc_auc_score(y_true=y_test, y_score=y_pred_proba_cat_nb[:, 1])

In [73]:
# roc_auc_score(y_true=y_test, y_score=y_pred_proba_gaussian_nb[:, 1])

In [74]:
# roc_auc_score(y_true=y_test, y_score=y_pred_mixed_nb[:, 1])

# ElasticNet

In [465]:
logreg = Pipeline([
    ('std_scaler', StandardScaler()),  # only for numerical purposes
    ('logreg', LogisticRegression(max_iter=1000, penalty='elasticnet', solver='saga', C=0.005, l1_ratio=0.01))
])

In [466]:
logreg.fit(X_train, y_train)

Pipeline(steps=[('std_scaler', StandardScaler()),
                ('logreg',
                 LogisticRegression(C=0.005, l1_ratio=0.01, max_iter=1000,
                                    penalty='elasticnet', solver='saga'))])

In [467]:
roc_auc_score(y_true=y_test, y_score=logreg.predict_proba(X_test)[:, 1])

0.7316570327552987

In [78]:
logreg.fit(X_data, y_data)

df_result = pd.DataFrame({'mid' : hidden_list, 'radiant_won' : logreg.predict_proba(X_hidden)[:, 1]}).set_index('mid')
df_result.to_csv('result.csv')

In [475]:
logreg_kfold = GridSearchCV(
    estimator=Pipeline([
        ('std_scaler', StandardScaler()), 
        ('logreg', LogisticRegression(max_iter=1000, penalty='elasticnet', solver='saga'))
    ]), 
    param_grid={'logreg__C' : np.logspace(-5, 1, 7), 'logreg__l1_ratio' : np.logspace(-5, 0, 6)}, 
    scoring=make_scorer(roc_auc_score),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=0),
    n_jobs=-1,
    verbose=0
)

logreg_kfold.fit(X=X_train, y=y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=0, shuffle=True),
             estimator=Pipeline(steps=[('std_scaler', StandardScaler()),
                                       ('logreg',
                                        LogisticRegression(max_iter=1000,
                                                           penalty='elasticnet',
                                                           solver='saga'))]),
             n_jobs=-1,
             param_grid={'logreg__C': array([1.e-05, 1.e-04, 1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01]),
                         'logreg__l1_ratio': array([1.e-05, 1.e-04, 1.e-03, 1.e-02, 1.e-01, 1.e+00])},
             scoring=make_scorer(roc_auc_score, response_method='predict'))

In [484]:
logreg_kfold.best_params_

{'logreg__C': 0.01, 'logreg__l1_ratio': 0.1}

: 

In [477]:
y_pred_proba_logreg = logreg_kfold.best_estimator_.predict_proba(X_test)

In [478]:
roc_auc_score(y_true=y_test, y_score=y_pred_proba_logreg[:, 1])

0.7332880539499036

In [479]:
y_pred_proba_hidden = logreg_kfold.best_estimator_.predict_proba(X_hidden)

In [480]:
radiant_won = y_pred_proba_hidden[:, 1]
radiant_won

array([0.56498508, 0.78621458, 0.14845962, ..., 0.49072555, 0.41029716,
       0.87376186])

In [481]:
# train on whole ds
logreg = Pipeline([
    ('std_scaler', StandardScaler()), 
    ('logreg', LogisticRegression(max_iter=3000, penalty='elasticnet', solver='saga'))
])

logreg.set_params(**logreg_kfold.best_params_)

logreg.fit(X_data, y_data)

Pipeline(steps=[('std_scaler', StandardScaler()),
                ('logreg',
                 LogisticRegression(C=0.01, l1_ratio=0.1, max_iter=3000,
                                    penalty='elasticnet', solver='saga'))])

In [482]:
roc_auc_score(y_true=y_train, y_score=logreg.predict_proba(X_train)[:, 1])

0.7927426072678806

In [483]:
df_result = pd.DataFrame({'mid' : hidden_list, 'radiant_won' : logreg.predict_proba(X_hidden)[:, 1]}).set_index('mid')
df_result.to_csv('result_en_cv.csv')

# SVC

In [39]:
from sklearn.svm import SVC, LinearSVC

In [40]:
svc_kfold = GridSearchCV(
    estimator=Pipeline([
        ('std_scaler', StandardScaler()), 
        ('svc', SVC(kernel='rbf'))
    ]), 
    param_grid={'svc__C' : np.logspace(0, 1, 1)}, 
    scoring=make_scorer(roc_auc_score),
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=0),
    n_jobs=1,
    verbose=3
)

In [68]:
# svc_kfold = GridSearchCV(
#     estimator=Pipeline([
#         ('std_scaler', StandardScaler()), 
#         ('svc', LinearSVC(penalty='l2', max_iter=3000))
#     ]), 
#     param_grid={'svc__C' : np.logspace(-5, 1, 6)}, 
#     scoring=make_scorer(roc_auc_score),
#     cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=0),
#     n_jobs=1,
#     verbose=3
# )

In [201]:
svc_kfold.fit(X_train, y_train)

Fitting 3 folds for each of 1 candidates, totalling 3 fits
[CV 1/3] END ........................svc__C=1.0;, score=0.661 total time= 3.5min
[CV 2/3] END ........................svc__C=1.0;, score=0.661 total time= 3.1min


KeyboardInterrupt: 

In [70]:
roc_auc_score(y_test, svc_kfold.best_estimator_.predict(X_test))

0.6459417148362235

# Random Forest

In [468]:
from sklearn.ensemble import RandomForestClassifier

In [472]:
rand_forest = RandomForestClassifier(
    n_estimators=1000,
    criterion='gini',
#    max_features=10,
    max_depth=5,
    n_jobs=-1
)

In [473]:
rand_forest.fit(X_train, y_train)

RandomForestClassifier(max_depth=5, n_estimators=1000, n_jobs=-1)

In [474]:
roc_auc_score(y_true=y_test, y_score=rand_forest.predict_proba(X_test)[:, 1])

0.6710470456005138